#### This notebook demonstrates the use of adversarial debiasing algorithm to learn a fair classifier.
Adversarial debiasing [1] is an in-processing technique that learns a classifier to maximize prediction accuracy and simultaneously reduce an adversary's ability to determine the protected attribute from the predictions. This approach leads to a fair classifier as the predictions cannot carry any group discrimination information that the adversary can exploit. We will see how to use this algorithm for learning models with and without fairness constraints and apply them on the Adult dataset.

In [16]:
! pip install 'aif360[Reductions]'
! pip install 'aif360[inFairness]'
# Create a directory to store all datasets
! mkdir -p AIF360_Datasets

# 1. Download Adult Census Dataset
! mkdir -p AIF360_Datasets/adult
! wget https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data -P AIF360_Datasets/adult/
! wget https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test -P AIF360_Datasets/adult/
# Make the target directory
! mkdir -p /usr/local/lib/python3.11/dist-packages/aif360/data/raw/adult

# Copy your downloaded files there
! cp AIF360_Datasets/adult/adult.data /usr/local/lib/python3.11/dist-packages/aif360/data/raw/adult/
! cp AIF360_Datasets/adult/adult.test /usr/local/lib/python3.11/dist-packages/aif360/data/raw/adult/

# Also download the 'adult.names' file there
! wget https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.names \
    -P /usr/local/lib/python3.11/dist-packages/aif360/data/raw/adult/


zsh:1: command not found: wget
zsh:1: command not found: wget
mkdir: /usr/local/lib: Permission denied
cp: directory /usr/local/lib/python3.11/dist-packages/aif360/data/raw/adult does not exist
cp: directory /usr/local/lib/python3.11/dist-packages/aif360/data/raw/adult does not exist
zsh:1: command not found: wget


The ! at the start means this is a shell command (executed in the terminal, not Python).

pip install installs Python packages.

The [Reductions] and [inFairness] are optional extras:

Reductions: installs extra algorithms (like Exponentiated Gradient, Grid Search) that reduce unfairness during training.

inFairness: installs fairness-related in-processing methods (like adversarial debiasing).

So here we’re installing the toolkit plus extra fairness algorithms.

The Adult dataset predicts whether someone earns >50k or <=50k income, based on attributes like age, sex, race, education, etc. It’s often used to study fairness because it contains sensitive attributes (gender, race).

In [2]:
%matplotlib inline
# Load all necessary packages
import sys
sys.path.append("../")
from aif360.datasets import BinaryLabelDataset
from aif360.datasets import AdultDataset, GermanDataset, CompasDataset
from aif360.metrics import BinaryLabelDatasetMetric
from aif360.metrics import ClassificationMetric
from aif360.metrics.utils import compute_boolean_conditioning_vector

from aif360.algorithms.preprocessing.optim_preproc_helpers.data_preproc_functions import load_preproc_data_adult, load_preproc_data_compas, load_preproc_data_german

#This imports the Adversarial Debiasing algorithm — an in-processing method that trains a classifier while trying to remove bias using an adversary network (like in GANs).
#It uses TensorFlow, which is why TF is imported later.
from aif360.algorithms.inprocessing.adversarial_debiasing import AdversarialDebiasing

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, MaxAbsScaler
from sklearn.metrics import accuracy_score

from IPython.display import Markdown, display
import matplotlib.pyplot as plt

import tensorflow.compat.v1 as tf
tf.disable_eager_execution()

/opt/anaconda3/envs/aif360/lib/python3.10/site-packages/inFairness/utils/ndcg.py:37: FutureWarning: We've integrated functorch into PyTorch. As the final step of the integration, `functorch.vmap` is deprecated as of PyTorch 2.0 and will be deleted in a future version of PyTorch >= 2.3. Please use `torch.vmap` instead; see the PyTorch 2.0 release notes and/or the `torch.func` migration guide for more details https://pytorch.org/docs/main/func.migrating.html
  vect_normalized_discounted_cumulative_gain = vmap(
/opt/anaconda3/envs/aif360/lib/python3.10/site-packages/inFairness/utils/ndcg.py:48: FutureWarning: We've integrated functorch into PyTorch. As the final step of the integration, `functorch.vmap` is deprecated as of PyTorch 2.0 and will be deleted in a future version of PyTorch >= 2.3. Please use `torch.vmap` instead; see the PyTorch 2.0 release notes and/or the `torch.func` migration guide for more details https://pytorch.org/docs/main/func.migrating.html
  monte_carlo_vect_ndcg =

BinaryLabelDataset: the main AIF360 data structure for datasets with binary labels (e.g. "yes/no", ">50K/<=50K").

AdultDataset, GermanDataset, CompasDataset: prebuilt dataset loaders for 3 common datasets used in fairness studies.

These help load and prepare datasets with fairness-related metadata (e.g., protected attributes like race, gender).

BinaryLabelDatasetMetric: computes dataset-level fairness metrics (like statistical parity, disparate impact) before training.

ClassificationMetric: computes fairness metrics after classification (on model predictions).

compute_boolean_conditioning_vector: utility to apply fairness metrics conditionally (e.g., only for a subgroup).

These measure how fair (or biased) your dataset and model are.

LogisticRegression: simple but interpretable classifier.

StandardScaler and MaxAbsScaler: normalize or scale data features.

accuracy_score: metric to compute the accuracy of predictions.

These are standard ML tools used for baseline modeling and comparison.

AdversarialDebiasing uses TensorFlow v1 style graph execution.

In TF v2, eager execution is on by default — so we disable it to stay compatible with AIF360’s TensorFlow code.

#### Load dataset and set options

In [19]:
# Get the dataset and split into train and test
dataset_orig = load_preproc_data_adult()

privileged_groups = [{'sex': 1}]
unprivileged_groups = [{'sex': 0}]

dataset_orig_train, dataset_orig_test = dataset_orig.split([0.7], shuffle=True)

/opt/anaconda3/envs/aif360/lib/python3.10/site-packages/aif360/algorithms/preprocessing/optim_preproc_helpers/data_preproc_functions.py:50: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['sex'] = df['sex'].replace({'Female': 0.0, 'Male': 1.0})


In [4]:
# print out some labels, names, etc.
display(Markdown("#### Training Dataset shape"))
print(dataset_orig_train.features.shape)
display(Markdown("#### Favorable and unfavorable labels"))
print(dataset_orig_train.favorable_label, dataset_orig_train.unfavorable_label)
display(Markdown("#### Protected attribute names"))
print(dataset_orig_train.protected_attribute_names)
display(Markdown("#### Privileged and unprivileged protected attribute values"))
print(dataset_orig_train.privileged_protected_attributes,
      dataset_orig_train.unprivileged_protected_attributes)
display(Markdown("#### Dataset feature names"))
print(dataset_orig_train.feature_names)

#### Training Dataset shape

(34189, 18)


#### Favorable and unfavorable labels

1.0 0.0


#### Protected attribute names

['sex', 'race']


#### Privileged and unprivileged protected attribute values

[array([1.]), array([1.])] [array([0.]), array([0.])]


#### Dataset feature names

['race', 'sex', 'Age (decade)=10', 'Age (decade)=20', 'Age (decade)=30', 'Age (decade)=40', 'Age (decade)=50', 'Age (decade)=60', 'Age (decade)=>=70', 'Education Years=6', 'Education Years=7', 'Education Years=8', 'Education Years=9', 'Education Years=10', 'Education Years=11', 'Education Years=12', 'Education Years=<6', 'Education Years=>12']


Datasets in AIF360 explicitly mark which label is “good” (favorable) and which is “bad” (unfavorable).

For example: in the Adult dataset,

Favorable = income > 50K → usually represented as 1

Unfavorable = income ≤ 50K → usually 0

Why? → Fairness metrics often check if unprivileged groups are equally likely to get the favorable outcome.

.favorable_label, .unfavorable_label, .protected_attribute_names, .privileged_protected_attributes, .unprivileged_protected_attributes - these attributes are metadata fields that the AIF360 BinaryLabelDataset object carries along with the raw data. They help us understand and work with fairness-related attributes.

.favorable_label: This tells you which label value is considered the “good outcome” in the dataset.

.protected_attribute_names: A list of attribute(s) in the dataset that are considered protected under fairness analysis.

.privileged_protected_attributes: This tells you which value(s) of the protected attribute correspond to the privileged group.

.favorable_label = 1 → high income (>50K)

.unfavorable_label = 0 → low income (≤50K)

.protected_attribute_names = ['sex']

.privileged_protected_attributes = [array([1.])] → males

.unprivileged_protected_attributes = [array([0.])] → females

These fields are defined when the dataset is preprocessed by load_preproc_data_adult(). That function hardcodes “sex” as the protected attribute by default (but you can change it).

#### Metric for original training data

In [5]:
# Metric for the original dataset
metric_orig_train = BinaryLabelDatasetMetric(dataset_orig_train,
                                             unprivileged_groups=unprivileged_groups,
                                             privileged_groups=privileged_groups)
display(Markdown("#### Original training dataset"))
print("Train set: Difference in mean outcomes between unprivileged and privileged groups = %f" % metric_orig_train.mean_difference())
metric_orig_test = BinaryLabelDatasetMetric(dataset_orig_test,
                                             unprivileged_groups=unprivileged_groups,
                                             privileged_groups=privileged_groups)
print("Test set: Difference in mean outcomes between unprivileged and privileged groups = %f" % metric_orig_test.mean_difference())

#### Original training dataset

Train set: Difference in mean outcomes between unprivileged and privileged groups = -0.195051
Test set: Difference in mean outcomes between unprivileged and privileged groups = -0.193280


This block calculates how unfair the dataset is before training any model.

If the dataset already shows strong bias, then even a “neutral” ML model will likely learn and replicate that bias.

Here, .mean_difference() computes: Mean outcome (unprivileged) − Mean outcome (privileged)

Negative value means privileged group gets more positives.
Positive value means unprivileged group gets more positives (rare, but possible).

In [6]:
min_max_scaler = MaxAbsScaler()
dataset_orig_train.features = min_max_scaler.fit_transform(dataset_orig_train.features)
dataset_orig_test.features = min_max_scaler.transform(dataset_orig_test.features)
metric_scaled_train = BinaryLabelDatasetMetric(dataset_orig_train,
                             unprivileged_groups=unprivileged_groups,
                             privileged_groups=privileged_groups)
display(Markdown("#### Scaled dataset - Verify that the scaling does not affect the group label statistics"))
print("Train set: Difference in mean outcomes between unprivileged and privileged groups = %f" % metric_scaled_train.mean_difference())
metric_scaled_test = BinaryLabelDatasetMetric(dataset_orig_test,
                             unprivileged_groups=unprivileged_groups,
                             privileged_groups=privileged_groups)
print("Test set: Difference in mean outcomes between unprivileged and privileged groups = %f" % metric_scaled_test.mean_difference())


#### Scaled dataset - Verify that the scaling does not affect the group label statistics

Train set: Difference in mean outcomes between unprivileged and privileged groups = -0.195051
Test set: Difference in mean outcomes between unprivileged and privileged groups = -0.193280


MaxAbsScaler scales each feature to be in the range [-1, 1] by dividing by the maximum absolute value in that feature.

Metrics are re-computed after scaling. Specifically: mean_difference = difference in positive outcomes between privileged vs unprivileged groups. Since scaling should not affect group membership or labels, the difference should stay the same as before.

### Learn plan classifier without debiasing

In [7]:
# Load post-processing algorithm that equalizes the odds
# Learn parameters with debias set to False
sess = tf.Session()
plain_model = AdversarialDebiasing(privileged_groups = privileged_groups,
                          unprivileged_groups = unprivileged_groups,
                          scope_name='plain_classifier',
                          debias=False,
                          sess=sess)

sess = tf.Session(): 
This creates a TensorFlow session (this is TensorFlow 1.x style).
The AdversarialDebiasing model internally uses TensorFlow graphs, so you need to provide a session where training will happen.

AdversarialDebiasing:
This is a classifier from AI Fairness 360 (AIF360).
It is essentially a neural network trained adversarially:
One part learns to predict the target label (e.g., income).
Another part tries to predict the protected attribute (e.g., sex).
The adversary forces the predictor to make predictions independent of the protected attribute, reducing bias.
(A neural network is a type of machine learning model inspired by how the human brain works. It’s especially good at finding complex patterns in data, like images, speech, or text.)

privileged_groups and unprivileged_groups:
Tell the model which group is considered "privileged" and which is "unprivileged".

debias=False → Just train a regular classifier (baseline, no fairness intervention).
debias=True → Train the fair adversarial classifier (the debiasing part is active).
That way, you can later compare the two models and see how much fairness improves when debiasing is turned on.

In [8]:
plain_model.fit(dataset_orig_train)

Instructions for updating:
Please use `rate` instead of `keep_prob`. Rate should be set to `rate = 1 - keep_prob`.
epoch 0; iter: 0; batch classifier loss: 0.656901
epoch 0; iter: 200; batch classifier loss: 0.431459


2025-08-16 19:59:33.400045: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:388] MLIR V1 optimization pass is not enabled


epoch 1; iter: 0; batch classifier loss: 0.532390
epoch 1; iter: 200; batch classifier loss: 0.411455
epoch 2; iter: 0; batch classifier loss: 0.432197
epoch 2; iter: 200; batch classifier loss: 0.460837
epoch 3; iter: 0; batch classifier loss: 0.468179
epoch 3; iter: 200; batch classifier loss: 0.405645
epoch 4; iter: 0; batch classifier loss: 0.389867
epoch 4; iter: 200; batch classifier loss: 0.456983
epoch 5; iter: 0; batch classifier loss: 0.421363
epoch 5; iter: 200; batch classifier loss: 0.431296
epoch 6; iter: 0; batch classifier loss: 0.402360
epoch 6; iter: 200; batch classifier loss: 0.417664
epoch 7; iter: 0; batch classifier loss: 0.510484
epoch 7; iter: 200; batch classifier loss: 0.408064
epoch 8; iter: 0; batch classifier loss: 0.403756
epoch 8; iter: 200; batch classifier loss: 0.450432
epoch 9; iter: 0; batch classifier loss: 0.466850
epoch 9; iter: 200; batch classifier loss: 0.510529
epoch 10; iter: 0; batch classifier loss: 0.373706
epoch 10; iter: 200; batch clas

plain_model: 
This is the AdversarialDebiasing defined earlier.
Internally, it builds a neural network classifier (TensorFlow-based).

What it learns depends on debias setting:

If debias=False → it learns like a normal neural net classifier (only focusing on accuracy).

If debias=True → it also trains an adversary network that tries to detect sensitive attributes (like sex, race).

The classifier is penalized if the adversary can guess the protected attribute well → making the final model more fair.

In [9]:
# Apply the plain model to test data
dataset_nodebiasing_train = plain_model.predict(dataset_orig_train)
dataset_nodebiasing_test = plain_model.predict(dataset_orig_test)

plain_model.predict(...): Runs the trained neural net on the input dataset.

dataset_nodebiasing_train: Predictions on the training set (dataset_orig_train).

dataset_nodebiasing_test: Predictions on the test set (dataset_orig_test). This is what really matters → shows how well your model generalizes to unseen data.

In [10]:
# Metrics for the dataset from plain model (without debiasing)
display(Markdown("#### Plain model - without debiasing - dataset metrics"))
metric_dataset_nodebiasing_train = BinaryLabelDatasetMetric(dataset_nodebiasing_train,
                                             unprivileged_groups=unprivileged_groups,
                                             privileged_groups=privileged_groups)

print("Train set: Difference in mean outcomes between unprivileged and privileged groups = %f" % metric_dataset_nodebiasing_train.mean_difference())

metric_dataset_nodebiasing_test = BinaryLabelDatasetMetric(dataset_nodebiasing_test,
                                             unprivileged_groups=unprivileged_groups,
                                             privileged_groups=privileged_groups)

print("Test set: Difference in mean outcomes between unprivileged and privileged groups = %f" % metric_dataset_nodebiasing_test.mean_difference())

display(Markdown("#### Plain model - without debiasing - classification metrics"))
classified_metric_nodebiasing_test = ClassificationMetric(dataset_orig_test,
                                                 dataset_nodebiasing_test,
                                                 unprivileged_groups=unprivileged_groups,
                                                 privileged_groups=privileged_groups)
print("Test set: Classification accuracy = %f" % classified_metric_nodebiasing_test.accuracy())
TPR = classified_metric_nodebiasing_test.true_positive_rate()
TNR = classified_metric_nodebiasing_test.true_negative_rate()
bal_acc_nodebiasing_test = 0.5*(TPR+TNR)
print("Test set: Balanced classification accuracy = %f" % bal_acc_nodebiasing_test)
print("Test set: Disparate impact = %f" % classified_metric_nodebiasing_test.disparate_impact())
print("Test set: Equal opportunity difference = %f" % classified_metric_nodebiasing_test.equal_opportunity_difference())
print("Test set: Average odds difference = %f" % classified_metric_nodebiasing_test.average_odds_difference())
print("Test set: Theil_index = %f" % classified_metric_nodebiasing_test.theil_index())

#### Plain model - without debiasing - dataset metrics

Train set: Difference in mean outcomes between unprivileged and privileged groups = -0.205010
Test set: Difference in mean outcomes between unprivileged and privileged groups = -0.216404


#### Plain model - without debiasing - classification metrics

Test set: Classification accuracy = 0.803248
Test set: Balanced classification accuracy = 0.662279
Test set: Disparate impact = 0.000000
Test set: Equal opportunity difference = -0.462210
Test set: Average odds difference = -0.285509
Test set: Theil_index = 0.178071


### Apply in-processing algorithm based on adversarial learning

In [11]:
sess.close()
tf.reset_default_graph()
sess = tf.Session()

In [12]:
# Learn parameters with debias set to True
debiased_model = AdversarialDebiasing(privileged_groups = privileged_groups,
                          unprivileged_groups = unprivileged_groups,
                          scope_name='debiased_classifier',
                          debias=True,
                          sess=sess)

In [13]:
debiased_model.fit(dataset_orig_train)

epoch 0; iter: 0; batch classifier loss: 0.715663; batch adversarial loss: 0.635791
epoch 0; iter: 200; batch classifier loss: 0.505382; batch adversarial loss: 0.645773
epoch 1; iter: 0; batch classifier loss: 0.540723; batch adversarial loss: 0.677794
epoch 1; iter: 200; batch classifier loss: 0.466853; batch adversarial loss: 0.636160
epoch 2; iter: 0; batch classifier loss: 0.508790; batch adversarial loss: 0.632413
epoch 2; iter: 200; batch classifier loss: 0.421913; batch adversarial loss: 0.637821
epoch 3; iter: 0; batch classifier loss: 0.372360; batch adversarial loss: 0.577181
epoch 3; iter: 200; batch classifier loss: 0.368119; batch adversarial loss: 0.637447
epoch 4; iter: 0; batch classifier loss: 0.475705; batch adversarial loss: 0.605564
epoch 4; iter: 200; batch classifier loss: 0.429074; batch adversarial loss: 0.573260
epoch 5; iter: 0; batch classifier loss: 0.294832; batch adversarial loss: 0.628211
epoch 5; iter: 200; batch classifier loss: 0.392057; batch adversa

What the values mean:

Classifier loss ↓ → The model is getting better at its main task (e.g., classification).

Adversarial loss ↑ or steady → The adversary struggles to predict the protected attribute → your model is becoming fairer.

In [14]:
# Apply the plain model to test data
dataset_debiasing_train = debiased_model.predict(dataset_orig_train)
dataset_debiasing_test = debiased_model.predict(dataset_orig_test)

In [15]:
# Metrics for the dataset from plain model (without debiasing)
display(Markdown("#### Plain model - without debiasing - dataset metrics"))
print("Train set: Difference in mean outcomes between unprivileged and privileged groups = %f" % metric_dataset_nodebiasing_train.mean_difference())
print("Test set: Difference in mean outcomes between unprivileged and privileged groups = %f" % metric_dataset_nodebiasing_test.mean_difference())

# Metrics for the dataset from model with debiasing
display(Markdown("#### Model - with debiasing - dataset metrics"))
metric_dataset_debiasing_train = BinaryLabelDatasetMetric(dataset_debiasing_train,
                                             unprivileged_groups=unprivileged_groups,
                                             privileged_groups=privileged_groups)

print("Train set: Difference in mean outcomes between unprivileged and privileged groups = %f" % metric_dataset_debiasing_train.mean_difference())

metric_dataset_debiasing_test = BinaryLabelDatasetMetric(dataset_debiasing_test,
                                             unprivileged_groups=unprivileged_groups,
                                             privileged_groups=privileged_groups)

print("Test set: Difference in mean outcomes between unprivileged and privileged groups = %f" % metric_dataset_debiasing_test.mean_difference())



display(Markdown("#### Plain model - without debiasing - classification metrics"))
print("Test set: Classification accuracy = %f" % classified_metric_nodebiasing_test.accuracy())
TPR = classified_metric_nodebiasing_test.true_positive_rate()
TNR = classified_metric_nodebiasing_test.true_negative_rate()
bal_acc_nodebiasing_test = 0.5*(TPR+TNR)
print("Test set: Balanced classification accuracy = %f" % bal_acc_nodebiasing_test)
print("Test set: Disparate impact = %f" % classified_metric_nodebiasing_test.disparate_impact())
print("Test set: Equal opportunity difference = %f" % classified_metric_nodebiasing_test.equal_opportunity_difference())
print("Test set: Average odds difference = %f" % classified_metric_nodebiasing_test.average_odds_difference())
print("Test set: Theil_index = %f" % classified_metric_nodebiasing_test.theil_index())



display(Markdown("#### Model - with debiasing - classification metrics"))
classified_metric_debiasing_test = ClassificationMetric(dataset_orig_test,
                                                 dataset_debiasing_test,
                                                 unprivileged_groups=unprivileged_groups,
                                                 privileged_groups=privileged_groups)
print("Test set: Classification accuracy = %f" % classified_metric_debiasing_test.accuracy())
TPR = classified_metric_debiasing_test.true_positive_rate()
TNR = classified_metric_debiasing_test.true_negative_rate()
bal_acc_debiasing_test = 0.5*(TPR+TNR)
print("Test set: Balanced classification accuracy = %f" % bal_acc_debiasing_test)
print("Test set: Disparate impact = %f" % classified_metric_debiasing_test.disparate_impact())
print("Test set: Equal opportunity difference = %f" % classified_metric_debiasing_test.equal_opportunity_difference())
print("Test set: Average odds difference = %f" % classified_metric_debiasing_test.average_odds_difference())
print("Test set: Theil_index = %f" % classified_metric_debiasing_test.theil_index())

#### Plain model - without debiasing - dataset metrics

Train set: Difference in mean outcomes between unprivileged and privileged groups = -0.205010
Test set: Difference in mean outcomes between unprivileged and privileged groups = -0.216404


#### Model - with debiasing - dataset metrics

Train set: Difference in mean outcomes between unprivileged and privileged groups = -0.070594
Test set: Difference in mean outcomes between unprivileged and privileged groups = -0.081849


#### Plain model - without debiasing - classification metrics

Test set: Classification accuracy = 0.803248
Test set: Balanced classification accuracy = 0.662279
Test set: Disparate impact = 0.000000
Test set: Equal opportunity difference = -0.462210
Test set: Average odds difference = -0.285509
Test set: Theil_index = 0.178071


#### Model - with debiasing - classification metrics

Test set: Classification accuracy = 0.789872
Test set: Balanced classification accuracy = 0.673495
Test set: Disparate impact = 0.615967
Test set: Equal opportunity difference = -0.045061
Test set: Average odds difference = -0.027712
Test set: Theil_index = 0.170023



    References:
    [1] B. H. Zhang, B. Lemoine, and M. Mitchell, "Mitigating UnwantedBiases with Adversarial Learning,"
    AAAI/ACM Conference on Artificial Intelligence, Ethics, and Society, 2018.